<a href="https://colab.research.google.com/github/awowy/spotify-audio-eda-numpy-pandas/blob/main/GDGOC1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Hands-On Task: Exploratory Data Analysis (EDA) - Spotify Dataset
**Module 1: Artificial Intelligence**

#### Task Description
This task aims to perform a comprehensive exploratory data analysis (EDA) on the Spotify song dataset using **NumPy** and **Pandas**. I will build a clean data pipeline, from initial data cleaning to in-depth audio feature analysis.

#### Work Plan - Stage 1
In this initial stage, I will execute the following steps:
1. Load the dataset directly from the provided GitHub URL.
2. Perform initial data inspection, including dimensions (shape), data types (dtypes), and missing value counts.
3. Execute data cleaning procedures, specifically handling missing values and duplicates with written justifications.

In [1]:
import pandas as pd
import numpy as np

# load the dataset directly from the source URL
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(url)

# stage 1.2: initial inspection
print("--- Dataset Information ---")
print(f"Shape: {df.shape}")
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- Missing Values Per Column ---")
print(df.isnull().sum())

--- Dataset Information ---
Shape: (32833, 23)

--- Data Types ---
track_id                     object
track_name                   object
track_artist                 object
track_popularity              int64
track_album_id               object
track_album_name             object
track_album_release_date     object
playlist_name                object
playlist_id                  object
playlist_genre               object
playlist_subgenre            object
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object

--- Missing Values Per Column ---
track_id                    0
track_name                

In [2]:
# calculate the threshold (20% of total rows)
threshold = 0.2 * len(df)

# identify the columns exceeding the threshold
cols_to_drop = df.columns[df.isnull().sum() > threshold].tolist()

if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped columns: {cols_to_drop}")
else:
    print("No columns exceeded the 20% missing value threshold.")

No columns exceeded the 20% missing value threshold.


In [3]:
# identify duplicates based on track_id
duplicate_count = df.duplicated(subset=['track_id']).sum()
print(f"Number of duplicate track IDs found: {duplicate_count}")

# the strategy is to move duplicates to ensure each song is unique in our analysis
df = df.drop_duplicates(subset=['track_id'], keep='first')

Number of duplicate track IDs found: 4477


In [4]:
# select only numeric columns for the summary table
numeric_df = df.select_dtypes(include=[np.number])

# initialize a dictionary to store our summary statistics
summary_stats = {
    "Mean": numeric_df.mean(),
    "Median": numeric_df.median(),
    "Std Dev": numeric_df.std()
}

# manual IQR Calculation using NumPy
# IQR = 75th percentile - 25th percentile
q75, q25 = np.percentile(numeric_df.dropna(), [75 ,25], axis=0)
summary_stats["IQR"] = q75 - q25

# create the final summary table
summary_table = pd.DataFrame(summary_stats)
print("\n--- Summary Table (Numeric Features) ---")
print(summary_table)


--- Summary Table (Numeric Features) ---
                           Mean         Median       Std Dev           IQR
track_popularity      39.329771      42.000000     23.702376     37.000000
danceability           0.653372       0.670000      0.145785      0.199000
energy                 0.698388       0.722000      0.183503      0.264000
key                    5.368000       6.000000      3.613904      7.000000
loudness              -6.817696      -6.261000      3.036243      3.600250
mode                   0.565489       1.000000      0.495701      1.000000
speechiness            0.107954       0.062600      0.102556      0.092000
acousticness           0.177176       0.079700      0.222803      0.245625
instrumentalness       0.091117       0.000021      0.232548      0.006570
liveness               0.190958       0.127000      0.155894      0.156400
valence                0.510387       0.512000      0.234340      0.366000
tempo                120.956180     121.993000     26.9545

I am using NumPy's nanpercentile function to manually calculate the IQR as required, ensuring that any null values do not break the vectorised calculation.

### Stage 2: Genre and Popularity Analysis
**Objective:** In this stage, I will segment the Spotify dataset by musical genres to uncover trends in track popularity and audio features. This analysis aims to provide actionable insights for content recommendation and artist evaluation.

#### The Key Goals:
1. **Genre Statistics:** Calculate the mean, standard deviation, and median for popularity and core audio features (danceability, energy, and valence) across all genres.
2. **Variance Analysis:** Identify which genre has the most unpredictable popularity and interpret the business implications for music curators.
3. **Artist Performance:** Evaluate the relationship between track volume and track quality (popularity) for the top 10 most prolific artists in the dataset.
4. **Hit Track Filtering:** Isolate high-performing tracks based on strict popularity and audio feature criteria to identify dominant trends.

In [5]:
# we use .agg() to apply multiple functions to multiple columns simultaneously
genre_analysis = df.groupby('playlist_genre')[['track_popularity', 'danceability', 'energy', 'valence']].agg(['mean', 'std', 'median'])

# diisplaying the resulting multi-index DataFrame
print("--- Distribution Statistics per Genre ---")
genre_analysis

--- Distribution Statistics per Genre ---


track_popularity                   danceability            \
                           mean        std median         mean       std   
playlist_genre                                                             
edm                   30.678286  20.346961   33.0     0.657639  0.123722   
latin                 41.439691  23.394529   45.0     0.711012  0.117222   
pop                   45.905300  24.616386   50.0     0.637698  0.128997   
r&b                   35.929396  23.662984   38.0     0.667475  0.137610   
rap                   41.822811  22.765566   46.0     0.715991  0.136217   
rock                  39.694309  24.229616   44.0     0.518519  0.140418   

                         energy                    valence                   
               median      mean       std median      mean       std median  
playlist_genre                                                               
edm             0.660  0.809604  0.136550  0.838  0.397491  0.228631  0.365  
latin           0.727  0.710468  0.156036  0.733  0.607390  0.225631  0.632  
pop             0.650  0.701031  0.172949  0.727  0.502176  0.221924  0.499  
r&b             0.687  0.588932  0.181949  0.594  0.537936  0.225879  0.548  
rap             0.734  0.649832  0.172107  0.666  0.505182  0.225269  0.517  
rock            0.522  0.733067  0.197484  0.779  0.532560  0.230204  0.526

In [6]:
# we access the 'std' column of 'track_popularity' from our previous genre_analysis table
popularity_std = genre_analysis[('track_popularity', 'std')].sort_values(ascending=False)

highest_variance_genre = popularity_std.idxmax()
highest_variance_value = popularity_std.max()

print(f"The genre with the highest variance in track popularity is: {highest_variance_genre}")
print(f"Standard Deviation value: {highest_variance_value:.2f}")

The genre with the highest variance in track popularity is: pop
Standard Deviation value: 24.62


Based on the standard deviation calculation of track_popularity across all genres, the Pop genre exhibits the highest variance with a standard deviation value of 24.62.

From a business and data science standpoint, a high variance in popularity within the Pop genre implies the following:
1. Pop is a "winner-takes-all" genre where a few songs become huge but most get ignored.
2. Streaming apps like Spotify cannot just recommend any random pop song because quality varies too much. They must look closely at specific details like beat and energy to keep listeners happy.
3. Curators find great viral stars in pop, but they have to filter through a lot of bad music to find them.

In [7]:
# step 1: find the 10 artists with the most tracks
top_10_artists_list = df['track_artist'].value_counts().head(10).index

# step 2: calculate the mean popularity for these 10 artists
artist_performance = df[df['track_artist'].isin(top_10_artists_list)].groupby('track_artist')['track_popularity'].mean().sort_values(ascending=False)

# step 3: identify the artist with the highest mean popularity among them
best_prolific_artist = artist_performance.idxmax()
best_popularity_score = artist_performance.max()

print("--- Top 10 Prolific Artists and Their Mean Popularity ---")
print(artist_performance)
print(f"\nThe artist with the highest mean popularity among the top 10 is: {best_prolific_artist} ({best_popularity_score:.2f})")

--- Top 10 Prolific Artists and Their Mean Popularity ---
track_artist
David Guetta                 49.370370
The Chainsmokers             49.227273
Queen                        42.400000
Logic                        41.907692
Drake                        41.808824
Martin Garrix                41.402299
Don Omar                     39.059524
Hardwell                     36.323529
Dimitri Vegas & Like Mike    36.220588
Guns N' Roses                27.206349
Name: track_popularity, dtype: float64

The artist with the highest mean popularity among the top 10 is: David Guetta (49.37)


**Analysis & Insights:**
Based on the results, the artist **David Guetta** has the highest mean popularity among the top 10 most prolific artists.

Regarding the correlation between volume and quality:
- In this dataset, a high volume of tracks does **not** necessarily guarantee high popularity.
- Many artists in the top 10 list have moderate popularity scores, suggesting that while they are productive, their tracks do not all become "hits."
- This suggests that "quality" (as measured by popularity) is independent of the sheer "quantity" of tracks an artist releases.

**Filtering Criteria:**
- **Popularity:** > 70
- **Danceability:** > 0.7
- **Energy:** > 0.6
- **Duration:** < 240,000 ms (4 minutes)

In [8]:
filtered_tracks = df[
    (df['track_popularity'] > 70) &
    (df['danceability'] > 0.7) &
    (df['energy'] > 0.6) &
    (df['duration_ms'] < 240000)
]

# count the number of tracks that pass the filter
track_count = len(filtered_tracks)

# identify the dominant genre in this high-potential subset
dominant_genre = filtered_tracks['playlist_genre'].value_counts()

print(f"Number of tracks passing the filters: {track_count}")
print("\n--- Genre Distribution of Potential Hits ---")
print(dominant_genre)

# identify the absolute dominant genre
top_genre = dominant_genre.idxmax()
print(f"\nThe dominant genre in this subset is: {top_genre}")

Number of tracks passing the filters: 579

--- Genre Distribution of Potential Hits ---
playlist_genre
pop      193
latin    173
rap      141
r&b       42
edm       19
rock      11
Name: count, dtype: int64

The dominant genre in this subset is: pop


**Task 8 Analysis:**
- **Result:** Out of the entire dataset, only **579** tracks met all the strict criteria for a "Potential Hit."
- **Dominant Genre:** The **pop** genre dominates this subset.
- **Musical Insight:** This result shows that tracks with high danceability and energy that are also popular tend to cluster within this specific genre. For a streaming business, this confirms that this genre is currently the most effective at producing high-engagement "radio-ready" content.

### Stage 3: NumPy Analysis
**Objective:** In this stage, i will transition from using Pandas to working directly with **NumPy arrays** to perform low-level mathematical operations. This is a critical step in a professional AI/ML workflow for data normalization and feature correlation analysis.

#### Key Technical Goals:
1. **Feature Extraction:** Isolate the nine core audio features into a pure NumPy array.
2. **Manual Min-Max Scaling:** Normalize the features to a [4] range using vectorized NumPy operations (without using loops or external libraries like sklearn).
3. **Correlation Analysis:** Compute a correlation matrix manually to understand the musical relationships between audio features.

In [9]:
import numpy as np

# 1. define the nine numeric audio feature columns
audio_features_cols = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# 2. xtract these columns as a NumPy array
# we use .values to convert the Pandas DataFrame slice into a NumPy array
audio_features_array = df[audio_features_cols].values

# 3. manual Min-Max Scaling (Vectorized)
# formula: (X - min) / (max - min)
# we calculate min and max along axis 0 (columns)
col_min = np.min(audio_features_array, axis=0)
col_max = np.max(audio_features_array, axis=0)

# apply the formula across the entire array at once
normalized_features = (audio_features_array - col_min) / (col_max - col_min)

# verification: Print the min and max of the normalized array
print(f"Shape of NumPy Array: {normalized_features.shape}")
print(f"Minimum value after scaling: {np.min(normalized_features)}")
print(f"Maximum value after scaling: {np.max(normalized_features)}")
print("\nFirst 3 rows of normalized data:")
print(normalized_features[:3])

Shape of NumPy Array: (28356, 9)
Minimum value after scaling: 0.0
Maximum value after scaling: 1.0

First 3 rows of normalized data:
[[7.60935910e-01 9.15985297e-01 9.18089810e-01 6.35076253e-02
  1.02615694e-01 0.00000000e+00 6.55622490e-02 5.22704339e-01
  5.09672569e-01]
 [7.38555443e-01 8.14967619e-01 8.69161620e-01 4.06318083e-02
  7.28370221e-02 4.23541247e-03 3.58433735e-01 6.99293643e-01
  4.17524223e-01]
 [6.86673449e-01 9.30987923e-01 9.01368313e-01 8.08278867e-02
  7.98792757e-02 2.34406439e-05 1.10441767e-01 6.18567104e-01
  5.17908453e-01]]


In [10]:
import numpy as np
import pandas as pd

# 1. compute the correlation matrix
# rowvar=False because our features are in columns
corr_matrix = np.corrcoef(normalized_features, rowvar=False)

# create a readable display using a DataFrame
corr_df = pd.DataFrame(corr_matrix, index=audio_features_cols, columns=audio_features_cols)

print("--- Correlation Matrix ---")
print(corr_df.round(2))

# 2. find the highest positive and most negative correlations
mask = np.eye(corr_matrix.shape[0], dtype=bool)

# we temporarily set the diagonal to 0 so we don't pick the 1.0 correlation of a feature with itself
corr_matrix_no_diag = np.where(mask, 0, corr_matrix)

# get the indices of the maximum and minimum values
max_pos_idx = np.unravel_index(np.argmax(corr_matrix_no_diag), corr_matrix_no_diag.shape)
max_neg_idx = np.unravel_index(np.argmin(corr_matrix_no_diag), corr_matrix_no_diag.shape)

# identify the names of the pairs
pair_pos = (audio_features_cols[max_pos_idx[0]], audio_features_cols[max_pos_idx[1]])
pair_neg = (audio_features_cols[max_neg_idx[0]], audio_features_cols[max_neg_idx[1]])

print(f"\nHighest Positive Correlation: {pair_pos[0]} & {pair_pos[1]} ({corr_matrix_no_diag[max_pos_idx]:.2f})")
print(f"Most Negative Correlation: {pair_neg[0]} & {pair_neg[1]} ({corr_matrix_no_diag[max_neg_idx]:.2f})")

--- Correlation Matrix ---
                  danceability  energy  loudness  speechiness  acousticness  \
danceability              1.00   -0.08      0.02         0.18         -0.03   
energy                   -0.08    1.00      0.68        -0.03         -0.55   
loudness                  0.02    0.68      1.00         0.01         -0.37   
speechiness               0.18   -0.03      0.01         1.00          0.02   
acousticness             -0.03   -0.55     -0.37         0.02          1.00   
instrumentalness         -0.00    0.02     -0.15        -0.11         -0.00   
liveness                 -0.13    0.16      0.08         0.06         -0.07   
valence                   0.33    0.15      0.05         0.06         -0.02   
tempo                    -0.18    0.15      0.10         0.03         -0.11   

                  instrumentalness  liveness  valence  tempo  
danceability                 -0.00     -0.13     0.33  -0.18  
energy                        0.02      0.16     0.15   

i will use NumPy boolean masking to identify "High-Energy Outliers" tracks with energy values exceeding one standard deviation above the mean ($\mu + \sigma$). I will then compare the average popularity of this energetic subset against the overall dataset average to see if extreme energy correlates with higher listener engagement.

In [11]:
import numpy as np

# 1. extract energy and popularity columns as pure NumPy arrays
energy_arr = df['energy'].values
popularity_arr = df['track_popularity'].values

# 2. calculate statistical thresholds
energy_mean = np.mean(energy_arr)
energy_std = np.std(energy_arr)
threshold = energy_mean + energy_std

# 3. create the boolean mask (NumPy-style)
# this generates an array of True/False values based on the condition
high_energy_mask = energy_arr > threshold

# 4. apply the mask to get the popularity values of the subset
high_energy_popularity = popularity_arr[high_energy_mask]

# 5. comparative analysis
overall_mean_pop = np.mean(popularity_arr)
subset_mean_pop = np.mean(high_energy_popularity)

print(f"Energy Mean (µ): {energy_mean:.3f}")
print(f"Energy Std Dev (σ): {energy_std:.3f}")
print(f"High Energy Threshold (µ + σ): {threshold:.3f}")
print(f"Number of High-Energy Tracks found: {len(high_energy_popularity)}")
print("-" * 30)
print(f"Overall Dataset Mean Popularity: {overall_mean_pop:.2f}")
print(f"High-Energy Subset Mean Popularity: {subset_mean_pop:.2f}")

Energy Mean (µ): 0.698
Energy Std Dev (σ): 0.183
High Energy Threshold (µ + σ): 0.882
Number of High-Energy Tracks found: 4853
------------------------------
Overall Dataset Mean Popularity: 39.33
High-Energy Subset Mean Popularity: 34.03


Statistical Findings:

Energy Mean (μ): 0.698

Energy Std Dev (σ): 0.183

High-Energy Threshold (μ+σ): 0.882

Number of High-Energy Tracks: 4,853

Overall Mean Popularity: 39.33

High-Energy Subset Mean Popularity: 34.03

Analytical Interpretation:

Popularity Gap: The data reveals that tracks with extreme energy levels (above 0.882) are, on average, 5.3 points less popular than the general dataset average.

Musical Insight: This suggests that while high energy is a core component of many genres, "extreme" energy might be associated with more niche or underground styles (such as Heavy Metal or Hardcore EDM) that do not reach the broad mainstream appeal of more "balanced" tracks.

Technical Reflection: By using NumPy Boolean Masking, I successfully filtered and analyzed nearly 5,000 specific data points out of 30,000+ rows instantaneously without the overhead of Python loops.

In [12]:
def calculate_manual_correlation(data_array, feature_names):
    # (the logic we built for task 10)
    corr_matrix = np.corrcoef(data_array, rowvar=False)
    mask = np.eye(corr_matrix.shape, dtype=bool)
    corr_matrix_no_diag = np.where(mask, 0, corr_matrix)

    max_pos_idx = np.unravel_index(np.argmax(corr_matrix_no_diag), corr_matrix_no_diag.shape)
    max_neg_idx = np.unravel_index(np.argmin(corr_matrix_no_diag), corr_matrix_no_diag.shape)

    return corr_matrix, max_pos_idx, max_neg_idx

def analyze_energy_outliers(energy_arr, popularity_arr):
    # (the logic we built for task 11)
    threshold = np.mean(energy_arr) + np.std(energy_arr)
    mask = energy_arr > threshold
    subset_popularity = popularity_arr[mask]
    return threshold, subset_popularity

# AI Tool Used: Gemini

### Prompt Used:
> "Write a professional Google-style docstring for this calculate_manual_correlation Python function. Include descriptions for arguments and return values."

---

### Evaluation of AI Output:
* **Accuracy:** The AI correctly identified the input types as `numpy.ndarray` and `list` of `str`. It also accurately described the return values, identifying that the function returns a tuple containing the correlation matrix and the specific indices for the highest/lowest correlations.
* **Context Awareness:** The AI correctly understood the logic of "excluding self-correlation (the diagonal)," which shows it grasped the mathematical intent of the code.

---

### Final Decision:
Used with minor modifications.

### Reasoning:
I accepted the docstring almost as-is because it meets the professional standards for Google-style documentation. However, I noticed that the AI's initial code suggestion for `np.eye` required a slight adjustment (shape) to handle the tuple input, which I implemented in the code to ensure it runs correctly without errors.


### Prompt Used:
> "Write a professional Google-style docstring for this analyze_energy_outliers Python function. Include descriptions for arguments and return values."

---

### Evaluation of AI Output:
* **Accuracy:** The AI successfully generated a perfectly aligned docstring for the `analyze_energy_outliers` function. It correctly identified the logic of using "one standard deviation above the mean" as the outlier threshold.
* **Technical Detail:** The AI accurately typed the arguments as `numpy.ndarray` and identified the return as a tuple containing a float and a `numpy.ndarray`.

---

### Final Decision:
Used as-is.

### Reasoning:
The generated docstring is concise, technically accurate, and adheres strictly to the Google-style documentation standards required for professional AI/ML projects.


#### Last Task: Three Key Non-Technical Insights
1. **Balanced Energy Wins:** Our data shows that while energy is key to many genres, "extreme" energy levels (the top outliers) are actually less popular on average. Listeners seem to prefer tracks that find a balance rather than pushing to statistical extremes.
2. **The Pop Paradox:** Pop music is the most unpredictable genre in the dataset. While it houses the biggest hits, it also has the highest variance, meaning it is a high-risk genre where success is never guaranteed by genre alone.
3. **Consistency over Quantity:** The most prolific artists—those with the highest number of tracks—do not necessarily have the highest popularity. This indicates that for Spotify's audience, a focused, high-quality discography is more impactful than sheer volume of releases.

## Bonus Section: Advanced Audio Analysis
**Objective:** To push beyond basic exploratory analysis, I will implement advanced machine learning concepts manually using NumPy.

### Bonus 1: Artist Audio Fingerprint & Cosine Similarity
**Goal:** i will create a "fingerprint" for specific artists based on the mean values of their 9 core audio features. I will then implement the mathematical definition of Cosine Similarity to compare how similar two artists are, regardless of their genre.


In [13]:
import numpy as np

# 1. define the artist fingerprint function
def artist_fingerprint(df, artist_name):
    """
    extracts the mean vector of the nine audio features for a specific artist.
    """
    features = ['danceability', 'energy', 'loudness', 'speechiness',
                'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

    # filter for the artist and calculate the mean for these 9 features
    artist_data = df[df['track_artist'] == artist_name][features]

    if artist_data.empty:
        return None

    return artist_data.mean().values

# 2. manual cosine similarity implementation
# formula: (A · B) / (||A|| * ||B||)
def calculate_cosine_similarity(vec_a, vec_b):
    if vec_a is None or vec_b is None:
        return 0

    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    return dot_product / (norm_a * norm_b)

# 3. comparing artist pairs
# let's compare some famous artists from the dataset
artists_to_compare = [
    ("Ed Sheeran", "Justin Bieber"), # Similar (Pop/Mainstream)
    ("Ed Sheeran", "Linkin Park"),   # Different (Pop vs Rock/Nu-Metal)
    ("Linkin Park", "Bring Me The Horizon") # Similar (Rock/Metalcore)
]

print("--- Artist Audio Fingerprint Comparison ---")
for artist_a, artist_b in artists_to_compare:
    fingerprint_a = artist_fingerprint(df, artist_a)
    fingerprint_b = artist_fingerprint(df, artist_b)

    similarity = calculate_cosine_similarity(fingerprint_a, fingerprint_b)
    print(f"Similarity between {artist_a} and {artist_b}: {similarity:.4f}")

--- Artist Audio Fingerprint Comparison ---
Similarity between Ed Sheeran and Justin Bieber: 0.9999
Similarity between Ed Sheeran and Linkin Park: 0.9997
Similarity between Linkin Park and Bring Me The Horizon: 0.9999


### Bonus 2: Genre Cluster Profiling and Euclidean Distance
**Objective:** i will construct an audio "profile" for every genre by calculating its centroid (mean feature vector). then, using numpy broadcasting, i will compute a complete pairwise distance matrix to mathematically identify which genres are musically closest and which are most distinct.

In [14]:
import numpy as np
import pandas as pd

# 1. compute centroids (mean feature vector per genre)
# we group by genre and take the mean of the 9 normalized features
genre_names = df['playlist_genre'].unique()
centroids = df.groupby('playlist_genre')[audio_features_cols].mean().values

# 2. compute pairwise euclidean distance matrix using broadcasting
# formula: sqrt(sum((a - b)^2))
# we expand dimensions to trigger numpy broadcasting (n_genres, 1, 9) - (1, n_genres, 9)
diff = centroids[:, np.newaxis, :] - centroids[np.newaxis, :, :]
dist_matrix = np.sqrt(np.sum(diff**2, axis=2))

# 3. convert to dataframe for better readability
dist_df = pd.DataFrame(dist_matrix, index=genre_names, columns=genre_names)

print("--- genre pairwise distance matrix ---")
print(dist_df.round(3))

--- genre pairwise distance matrix ---
          pop    rap   rock   latin    r&b     edm
pop     0.000  7.876  5.445  12.676  5.976   2.457
rap     7.876  0.000  2.446   4.862  2.150   6.609
rock    5.445  2.446  0.000   7.240  0.824   4.247
latin  12.676  4.862  7.240   0.000  6.730  11.135
r&b     5.976  2.150  0.824   6.730  0.000   4.493
edm     2.457  6.609  4.247  11.135  4.493   0.000


In [15]:
# 4. identify the most similar and most different genres
# we mask the diagonal (zero distance to itself)
mask = np.eye(len(genre_names), dtype=bool)

# for finding minimum distance, set diagonal to infinity
dist_matrix_for_argmin = np.where(mask, np.inf, dist_matrix)
min_idx = np.unravel_index(np.argmin(dist_matrix_for_argmin), dist_matrix_for_argmin.shape)

# for finding maximum distance, set diagonal to negative infinity
# this ensures np.argmax picks the largest finite value, not np.inf from the diagonal
dist_matrix_for_argmax = np.where(mask, -np.inf, dist_matrix)
max_idx = np.unravel_index(np.argmax(dist_matrix_for_argmax), dist_matrix_for_argmax.shape)

# access genre names using the two elements of the index tuple
print(f"\nmost similar genres: {genre_names[min_idx[0]]} & {genre_names[min_idx[1]]} (dist: {dist_matrix[min_idx]:.4f})")
print(f"most different genres: {genre_names[max_idx[0]]} & {genre_names[max_idx[1]]} (dist: {dist_matrix[max_idx]:.4f})")


most similar genres: rock & r&b (dist: 0.8243)
most different genres: pop & latin (dist: 12.6757)


In [16]:
def recommend_similar_genre(genre, top_k=3):
    """
    returns the top_k most similar genres based on the distance matrix.
    """
    if genre not in dist_df.index:
        return "genre not found."

    # get distances for the specific genre and sort them
    recommendations = dist_df[genre].sort_values().iloc[1:top_k+1]
    return recommendations

# testing the function
# iterate through each genre name
print("\n--- Genre Similarity Recommendations ---")
for target_genre_name in genre_names:
    print(f"\nTop 3 genres similar to '{target_genre_name}':")
    print(recommend_similar_genre(target_genre_name))


--- Genre Similarity Recommendations ---

Top 3 genres similar to 'pop':
edm     2.457157
rock    5.444767
r&b     5.976170
Name: pop, dtype: float64

Top 3 genres similar to 'rap':
r&b      2.149528
rock     2.446317
latin    4.862379
Name: rap, dtype: float64

Top 3 genres similar to 'rock':
r&b    0.824252
rap    2.446317
edm    4.246917
Name: rock, dtype: float64

Top 3 genres similar to 'latin':
rap     4.862379
r&b     6.730139
rock    7.239762
Name: latin, dtype: float64

Top 3 genres similar to 'r&b':
rock    0.824252
rap     2.149528
edm     4.493080
Name: r&b, dtype: float64

Top 3 genres similar to 'edm':
pop     2.457157
rock    4.246917
r&b     4.493080
Name: edm, dtype: float64


### Bonus 3: Professional Reusable Analysis Class
**objective:** i will refactor the entire eda pipeline into a structured python class named `spotifyanalyzer`. this class will incorporate full type hints, modular methods for each analysis stage, and a automated reporting function. this represents the transition from experimental scripting to production-ready ai/ml code.

In [17]:
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Tuple

class spotifyanalyzer:
    def __init__(self, url: str):
        # initialize the analyzer by loading the dataset from the provided url
        self.df = pd.read_csv(url)
        self.audio_features_cols = [
            'danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
        ]
        self.normalized_features = None

    def clean_data(self) -> None:
        # handle missing values and remove duplicates based on track name and artist
        self.df.dropna(subset=['track_name', 'track_artist', 'track_album_name'], inplace=True)
        self.df.drop_duplicates(subset=['track_name', 'track_artist'], keep='first', inplace=True)

    def analyze_genres(self) -> Dict[str, Any]:
        # compute statistics per genre and identify the high-variance genre
        stats = self.df.groupby('playlist_genre')['track_popularity'].agg(['mean', 'std', 'median'])
        highest_var_genre = stats['std'].idxmax()
        return {
            "highest_variance_genre": highest_var_genre,
            "variance_value": stats.loc[highest_var_genre, 'std']
        }

    def process_numpy_analysis(self) -> Tuple[np.ndarray, Dict[str, Any]]:
        # perform manual min-max scaling and find correlations using numpy
        raw_array = self.df[self.audio_features_cols].values
        col_min = np.min(raw_array, axis=0)
        col_max = np.max(raw_array, axis=0)
        self.normalized_features = (raw_array - col_min) / (col_max - col_min)

        # calculate manual correlation
        corr_matrix = np.corrcoef(self.normalized_features, rowvar=False)
        return self.normalized_features, {"correlation_matrix_shape": corr_matrix.shape}

    def generate_report(self) -> Dict[str, Any]:
        # compile all key insights into a single dictionary for final output
        genre_info = self.analyze_genres()

        # energy outlier check
        energy_arr = self.df['energy'].values
        threshold = np.mean(energy_arr) + np.std(energy_arr)
        high_energy_pop = self.df['track_popularity'].values[energy_arr > threshold].mean()
        overall_pop = self.df['track_popularity'].mean()

        return {
            "total_tracks_analyzed": len(self.df),
            "top_variance_genre": genre_info["highest_variance_genre"],
            "energy_outlier_popularity": round(high_energy_pop, 2),
            "overall_popularity": round(overall_pop, 2),
            "popularity_gap": round(overall_pop - high_energy_pop, 2)
        }

# demonstration: running the entire analysis in one go
dataset_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
analyzer = spotifyanalyzer(dataset_url)
analyzer.clean_data()
analyzer.process_numpy_analysis()
report = analyzer.generate_report()

print("--- spotify analyzer automated report ---")
import json
print(json.dumps(report, indent=4))

--- spotify analyzer automated report ---
{
    "total_tracks_analyzed": 26229,
    "top_variance_genre": "pop",
    "energy_outlier_popularity": 34.03,
    "overall_popularity": 39.51,
    "popularity_gap": 5.48
}


# AI Tool Used: Gemini

### Prompt Used:
> "write a comprehensive google-style docstring for this generate_report method, including return type details. [code provided]"a

---

### Evaluation of AI Output:
* **Technical Accuracy:** the ai correctly identified all dictionary keys returned by the method (total_tracks_analyzed, top_variance_genre, etc.) and mapped their data types correctly.
* **Depth of Explanation:** i particularly like how the ai added a "note" section mentioning the dependency on clean_data(). this shows the ai understood the class state and the need for a populated dataframe.
* **Formatting:** the docstring follows the indentation and section headers (args, returns, note) required by the google python style guide.

---

### Final Decision:
Used as is.

### Reasoning:
the output was of such high quality that no modifications were needed. it successfully translated the code logic into clear, human-readable documentation.
